In [3]:
# 07_DOCUMENTATION — PART1 / CELL 01
# REPOSITORY INVENTORY & DOCUMENTATION BLUEPRINT
#
# Exact Git HEAD | tracked-file inventory | SHA256
# Read-only research sources | No overwrite
# No retraining | Self-contained (stdlib + pandas)
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import subprocess
import hashlib
import json
import os
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
OUT = ROOT / "07_Documentation" / "outputs" / "repository_inventory"

EXPECTED_HEAD = (
    "ae5107c317eaee11a909251470be4b7503a6f9db"
)

AUDIT6 = (
    ROOT / "06_Result" / "outputs"
    / "final_technical_report"
    / "part6_final_technical_audit.json"
)

INVENTORY_OUT = OUT / "repository_inventory.csv"
SUMMARY_OUT = OUT / "repository_inventory_summary.json"
BLUEPRINT_OUT = OUT / "documentation_blueprint.md"
AUDIT_OUT = OUT / "part1_repository_inventory_audit.json"

OUTPUTS = [
    INVENTORY_OUT,
    SUMMARY_OUT,
    BLUEPRINT_OUT,
    AUDIT_OUT
]

EXPECTED_PHASES = [
    "04_Modeling",
    "05_Temporal",
    "06_Result"
]

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

def git_bytes(*args):
    p = subprocess.run(
        ["git", "-C", str(ROOT), *args],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return p.stdout

def git_text(*args):
    return git_bytes(*args).decode("utf-8").strip()

def json_bytes(data):
    return json.dumps(
        data, ensure_ascii=False, indent=2,
        allow_nan=False
    ).encode("utf-8")

def phase_of(path):
    p = Path(path)
    first = p.parts[0] if p.parts else ""
    if first in {
        "01_Raw Data", "01_Data_Collection",
        "01_DataCollection"
    }:
        return "01_Collection"
    if first.startswith("01_"):
        return "01_Collection"
    if first.startswith("02_"):
        return "02_Preprocessing"
    if first.startswith("03_"):
        return "03_Labeling"
    if first.startswith("04_"):
        return "04_Modeling"
    if first.startswith("05_"):
        return "05_Temporal"
    if first.startswith("06_"):
        return "06_Result"
    if first.startswith("07_"):
        return "07_Documentation"
    return "ROOT_OR_OTHER"

def artifact_type(path):
    suffix = Path(path).suffix.lower()
    return {
        ".ipynb": "notebook",
        ".csv": "dataset_or_table",
        ".json": "json_manifest_or_audit",
        ".md": "markdown_document",
        ".pdf": "pdf_report",
        ".png": "figure_png",
        ".jpg": "figure_jpg",
        ".jpeg": "figure_jpg",
        ".py": "python_script",
        ".txt": "text_or_config",
        ".toml": "environment_config",
        ".yml": "environment_config",
        ".yaml": "environment_config",
        ".lock": "dependency_lock"
    }.get(suffix, "other")

def role_of(path):
    lower = path.lower()
    if lower.endswith(".ipynb"):
        return "execution_notebook"
    if "audit" in lower:
        return "audit_evidence"
    if "manifest" in lower or "registry" in lower:
        return "provenance_or_manifest"
    if lower.endswith((".png", ".jpg", ".jpeg")):
        return "research_figure"
    if lower.endswith(".pdf"):
        return "report_or_reference"
    if lower.endswith(".csv"):
        return "dataset_or_results_table"
    if lower.endswith(".json"):
        return "structured_research_artifact"
    if lower.endswith(".md"):
        return "research_documentation"
    return "other"

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

print("=" * 76)
print("07_DOCUMENTATION PART1 — REPOSITORY PREFLIGHT")
print("=" * 76)

assert ROOT.is_dir(), f"Root missing: {ROOT}"
assert (ROOT / ".git").exists(), "Git repository not found."
assert AUDIT6.is_file(), "Result Part6 audit missing."

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing overwrite: {path}"
        )

# ------------------------------------------------------------
# A. GIT COMMIT LOCK
# ------------------------------------------------------------

head = git_text("rev-parse", "HEAD")
branch = git_text("branch", "--show-current")

assert head == EXPECTED_HEAD, (
    "Git HEAD differs from audited commit.\n"
    f"Expected: {EXPECTED_HEAD}\nActual: {head}\n"
    "Review repository changes before continuing."
)

# Normalize Windows paths before comparing Git repository roots.
git_root = Path(
    git_text("rev-parse", "--show-toplevel")
).resolve()

expected_root = ROOT.resolve()

assert os.path.normcase(str(git_root)) == os.path.normcase(
    str(expected_root)
), (
    "Git repository root mismatch.\n"
    f"Git root : {git_root}\n"
    f"Expected : {expected_root}"
)

print("GIT REPOSITORY ROOT: PASS")
print("Repository:", git_root)

# Worktree changes to committed research source files are
# forbidden. The new documentation notebook may be untracked.
status_raw = git_bytes(
    "status", "--porcelain=v1", "-z", "--untracked-files=no"
)

changed_tracked = []

for entry in status_raw.split(b"\0"):
    if not entry:
        continue

    decoded = entry.decode("utf-8", errors="replace")
    changed_tracked.append(decoded)

# Git rename/copy porcelain can use two NUL entries.
# Rather than parsing ambiguously, fail closed for any
# tracked change.
assert not changed_tracked, (
    "Tracked files have uncommitted changes. "
    "Commit or restore them before inventory.\n"
    + "\n".join(changed_tracked[:20])
)

print("GIT HEAD LOCK: PASS")
print("Commit:", head)
print("Branch:", branch)

# ------------------------------------------------------------
# B. VALIDATE RESULT PART6 SOURCE
# ------------------------------------------------------------

a6 = read_json(AUDIT6)

assert a6["status"] == (
    "RESULT_PART6_TECHNICAL_AUDIT_PASS"
)

assert a6["dataset_rows"] == 348
assert a6["research_questions"] == 3
assert a6["original_figures_in_pdf"] == 10
assert a6["methodological_issues_open"] == 4
assert a6["academic_status"] == "PROVISIONAL"
assert a6["supervisor_approval_confirmed"] is False
assert a6["model_training_performed"] is False

for source_name, expected_hash in (
    a6["source_sha256"].items()
):
    path = ROOT / source_name

    assert path.is_file(), f"Missing locked source: {path}"
    assert sha256(path) == expected_hash, (
        f"Locked source modified: {source_name}"
    )

for image_name, expected_hash in (
    a6["original_figure_sha256"].items()
):
    path = ROOT / image_name

    assert path.is_file(), f"Missing original image: {path}"
    assert sha256(path) == expected_hash, (
        f"Original image modified: {image_name}"
    )

final_dir = AUDIT6.parent

for filename, expected_hash in (
    a6["output_sha256"].items()
):
    path = final_dir / filename
    assert path.is_file(), f"Missing Part6 output: {path}"
    assert sha256(path) == expected_hash

audit6_hash = sha256(AUDIT6)

print("RESULT PART6 & LOCKED RESEARCH SOURCES: PASS")

# ------------------------------------------------------------
# C. INVENTORY ALL TRACKED FILES AT EXACT HEAD
# ------------------------------------------------------------

tracked_raw = git_bytes(
    "ls-tree", "-r", "-z", "--full-tree", "HEAD"
)

records = []
seen_paths = set()

for entry in tracked_raw.split(b"\0"):
    if not entry:
        continue

    metadata, path_bytes = entry.split(b"\t", 1)
    mode, kind, git_oid = metadata.decode("ascii").split()

    relative_path = path_bytes.decode("utf-8")
    assert relative_path not in seen_paths
    seen_paths.add(relative_path)

    # Fail closed on submodules, symlinks, or unusual Git types.
    assert kind == "blob", (
        f"Unsupported Git tree item: {relative_path}"
    )
    assert mode in {"100644", "100755"}, (
        f"Unexpected Git mode {mode}: {relative_path}"
    )

    file_path = (ROOT / relative_path).resolve()

    assert file_path.is_relative_to(ROOT.resolve()), (
        f"Unsafe path: {relative_path}"
    )
    assert file_path.is_file(), (
        f"Tracked file missing locally: {relative_path}"
    )
    assert not file_path.is_symlink()

    file_size = file_path.stat().st_size

    records.append({
        "repository_path": relative_path,
        "phase": phase_of(relative_path),
        "artifact_type": artifact_type(relative_path),
        "role": role_of(relative_path),
        "extension": file_path.suffix.lower(),
        "size_bytes": file_size,
        "sha256": sha256(file_path),
        "git_mode": mode,
        "git_blob_oid": git_oid
    })

inventory = pd.DataFrame(records)
inventory = inventory.sort_values(
    "repository_path"
).reset_index(drop=True)

assert len(inventory) > 0
assert inventory["repository_path"].is_unique
assert inventory["sha256"].str.len().eq(64).all()

phase_counts = Counter(inventory["phase"])

for phase in EXPECTED_PHASES:
    assert phase_counts[phase] > 0, (
        f"No tracked files in {phase}"
    )

# The inventory represents the Git HEAD tree exactly.
assert len(inventory) == len(seen_paths)

print("TRACKED GIT TREE INVENTORY: PASS")
print("Total tracked files:", len(inventory))

# ------------------------------------------------------------
# D. COMPLETENESS & CRITICAL ARTIFACT VALIDATION
# ------------------------------------------------------------

required_paths = [
    "04_Modeling/04_modeling.ipynb",
    "05_Temporal/05_temporal.ipynb",
    "06_Result/06_result.ipynb",
    "04_Modeling/outputs/comparative_evaluation/"
        "paired_oof_comparison_348.csv",
    "05_Temporal/outputs/daily_aggregation/"
        "daily_sentiment_counts_20_days.csv",
    "06_Result/outputs/integrated_report/"
        "provisional_integrated_research_report.md",
    "06_Result/outputs/final_technical_report/"
        "provisional_integrated_research_report.pdf",
    "06_Result/outputs/final_technical_report/"
        "part6_final_technical_audit.json"
]

path_set = set(inventory["repository_path"])

for path in required_paths:
    assert path in path_set, (
        f"Critical tracked artifact missing: {path}"
    )

source_inventory = inventory.set_index("repository_path")

assert source_inventory.loc[
    "06_Result/outputs/final_technical_report/"
    "part6_final_technical_audit.json", "sha256"
] == audit6_hash

# ------------------------------------------------------------
# E. DOCUMENTATION BLUEPRINT
# ------------------------------------------------------------

summary = {
    "stage": "07_Documentation",
    "part": "1 Repository Inventory",
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "repository": (
        "YusakSR/Penelitian-Sentimen-Bencana-Nasional"
    ),
    "head_commit": head,
    "head_branch": branch,
    "tracked_file_count": int(len(inventory)),
    "tracked_total_size_bytes": int(
        inventory["size_bytes"].sum()
    ),
    "phase_file_counts": dict(
        sorted(Counter(inventory["phase"]).items())
    ),
    "artifact_type_counts": dict(
        sorted(Counter(inventory["artifact_type"]).items())
    ),
    "extension_counts": dict(
        sorted(Counter(inventory["extension"]).items())
    ),
    "critical_artifacts_verified": len(required_paths),
    "result_part6_audit_sha256": audit6_hash,
    "academic_status": "PROVISIONAL",
    "methodological_issues_open": 4,
    "full_git_tree_inventory": True,
    "model_training_performed": False
}

phase_lines = "\n".join(
    f"| {phase} | {count} |"
    for phase, count in sorted(phase_counts.items())
)

blueprint = f"""# 07_DOCUMENTATION — BLUEPRINT

**Repository:** {summary["repository"]}

**Commit terkunci:** `{head}`

**Status akademik:** PROVISIONAL

**Jumlah berkas terlacak:** {len(inventory)}

## 1. Prinsip Dokumentasi

Fase ini menjelaskan pipeline dan bukti penelitian
tanpa mengubah raw data, label, model, prediksi OOF,
grafik, atau laporan sumber.

Setiap part memakai input terkunci, audit SHA-256,
dan tidak menimpa output yang sudah diterbitkan.

## 2. Inventaris Repository

| Fase / kategori | Berkas |
|---|---:|
{phase_lines}

Daftar lengkap tersedia pada
`repository_inventory.csv`.

Inventaris ini merepresentasikan berkas terlacak
Git pada commit yang disebutkan di atas.
Berkas lokal belum terlacak tidak termasuk.

## 3. Rencana Part2–Part7

### Part2 — Data Provenance & Collection Documentation

Dokumentasikan aliran data dari pengumpulan,
pembersihan, deduplikasi, hingga 348 tweet
berlabel yang menjadi dataset penelitian.

Artefak:
- Data lineage
- Data dictionary
- Sampling and labeling documentation
- Provenance audit

### Part3 — Modeling & Temporal Reproducibility

Dokumentasikan:
- Python environment dan dependencies
- Seed dan SGKF 3-fold
- Tokenisasi dan training LSTM
- Baseline TF-IDF + Logistic Regression
- Prediksi OOF
- Agregasi tanggal WIB
- Sumber tabel dan grafik terkunci

Artefak:
- Reproducibility guide
- Configuration registry
- Source manifest
- Audit

### Part4 — Academic Methodology Alignment

Verifikasi kecocokan proposal, RM1–RM3,
dan implementasi.

Empat isu harus tetap terbuka kecuali terdapat
keputusan eksplisit dan terdokumentasi
dari pembimbing.

### Part5 — Figures, Tables & Reporting Guide

Susun katalog tabel, gambar, caption,
sumber bukti, dan pedoman editorial BAB IV.

### Part6 — End-to-End Repository Integrity Audit

Periksa rantai hash dan konsistensi artefak
tanpa melakukan retraining atau scraping ulang.

### Part7 — Final Documentation Handoff

Susun panduan repository, ringkasan pipeline,
manifest, serta paket penyerahan penelitian.

## 4. Gerbang Kualitas

- Semua artefak sumber harus dapat diverifikasi.
- Tidak ada penulisan ulang file sumber.
- Tidak ada angka atau rumusan masalah yang dikarang.
- Dataset penelitian tetap 348 tweet.
- Keempat isu metodologis tetap dilaporkan.
- PDF BAB IV tetap berstatus PROVISIONAL.
- Pemeriksaan visual PDF harus diselesaikan
  sebelum penyerahan akademik.
- Perubahan metode memerlukan persetujuan pembimbing.

## 5. Output Part1

- `repository_inventory.csv`
- `repository_inventory_summary.json`
- `documentation_blueprint.md`
- `part1_repository_inventory_audit.json`

**Tahap berikutnya:** Documentation Part2 —
Data Provenance & Collection Documentation.
"""

# ------------------------------------------------------------
# F. OUTPUT GENERATION — NO OVERWRITE
# ------------------------------------------------------------

output_bytes = {
    INVENTORY_OUT: inventory.to_csv(
        index=False
    ).encode("utf-8-sig"),
    SUMMARY_OUT: json_bytes(summary),
    BLUEPRINT_OUT: blueprint.encode("utf-8")
}

audit1 = {
    "stage": "07_Documentation",
    "part": "1 Repository Inventory & Blueprint",
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "head_commit": head,
    "head_branch": branch,
    "head_tree_sha": git_text(
        "rev-parse", "HEAD^{tree}"
    ),
    "tracked_files": int(len(inventory)),
    "tracked_total_size_bytes": int(
        inventory["size_bytes"].sum()
    ),
    "critical_artifacts_verified": len(required_paths),
    "result_part6_status": a6["status"],
    "result_part6_audit_sha256": audit6_hash,
    "dataset_rows": 348,
    "methodological_issues_open": 4,
    "academic_status": "PROVISIONAL",
    "sources_modified": False,
    "model_training_performed": False,
    "output_sha256": {
        path.name: hashlib.sha256(data).hexdigest()
        for path, data in output_bytes.items()
    },
    "status": "DOC_PART1_REPOSITORY_INVENTORY_PASS"
}

output_bytes[AUDIT_OUT] = json_bytes(audit1)

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing overwrite: {path}"
        )

created = []

try:
    # Reconfirm commit has not changed during execution.
    assert git_text("rev-parse", "HEAD") == head

    assert sha256(AUDIT6) == audit6_hash

    for path in OUTPUTS:
        with path.open("xb") as f:
            f.write(output_bytes[path])
        created.append(path)

    for path, data in output_bytes.items():
        assert sha256(path) == hashlib.sha256(
            data
        ).hexdigest()

    assert len(
        pd.read_csv(
            INVENTORY_OUT, encoding="utf-8-sig"
        )
    ) == len(inventory)

    assert read_json(AUDIT_OUT)["status"] == (
        "DOC_PART1_REPOSITORY_INVENTORY_PASS"
    )

except Exception:
    for path in created:
        if path.exists():
            path.unlink()
    raise

# ------------------------------------------------------------
# G. FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("07_DOCUMENTATION PART1 — FINAL AUDIT")
print("=" * 76)

print("Git commit            :", head[:12])
print("Tracked files         :", len(inventory))
print("Critical files        :", len(required_paths))
print("Dataset rows          : 348")
print("Methodological issues : 4 OPEN")
print("Academic status       : PROVISIONAL")
print("Source modifications  : NO")
print("Model training        : NO")

print("\nPHASE INVENTORY")
display(
    inventory.groupby("phase")
    .agg(
        files=("repository_path", "count"),
        total_size_bytes=("size_bytes", "sum")
    )
    .reset_index()
)

print("\nOUTPUT FILES")
for path in OUTPUTS:
    print("-", path)

print("\nFINAL STATUS:", audit1["status"])
print("=" * 76)


07_DOCUMENTATION PART1 — REPOSITORY PREFLIGHT
GIT REPOSITORY ROOT: PASS
Repository: C:\Penelitian Sentimen Bencana Nasional
GIT HEAD LOCK: PASS
Commit: ae5107c317eaee11a909251470be4b7503a6f9db
Branch: main
RESULT PART6 & LOCKED RESEARCH SOURCES: PASS
TRACKED GIT TREE INVENTORY: PASS
Total tracked files: 455

07_DOCUMENTATION PART1 — FINAL AUDIT
Git commit            : ae5107c317ea
Tracked files         : 455
Critical files        : 8
Dataset rows          : 348
Methodological issues : 4 OPEN
Academic status       : PROVISIONAL
Source modifications  : NO
Model training        : NO

PHASE INVENTORY


,phase,files,total_size_bytes
0,01_Collection,251,22907894
1,02_Preprocessing,11,1634875
2,03_Labeling,33,2571988
3,04_Modeling,89,4138364
4,05_Temporal,26,2365800
5,06_Result,29,1429234
6,07_Documentation,1,2
7,ROOT_OR_OTHER,15,377896



OUTPUT FILES
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\repository_inventory.csv
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\repository_inventory_summary.json
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\documentation_blueprint.md
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\part1_repository_inventory_audit.json

FINAL STATUS: DOC_PART1_REPOSITORY_INVENTORY_PASS
